# Project 5 — JetAuto Navigation

<svg width="100%" viewBox="0 0 1260 150" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Navigation project workflow">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1260" height="150" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="30" font-family="Arial" font-size="20" font-weight="700" fill="#0f172a">Navigation project workflow</text>
<rect x="25.0" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="115.4" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Load map</text>
<line x1="205.8" y1="84" x2="224.8" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="230.8" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="321.2" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Initialize pose</text>
<line x1="411.7" y1="84" x2="430.7" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="436.7" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="527.1" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Run waypoints</text>
<line x1="617.5" y1="84" x2="636.5" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="642.5" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="732.9" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Record no-obstacle</text>
<text x="732.9" y="97" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">run</text>
<line x1="823.3" y1="84" x2="842.3" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="848.3" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="938.8" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Add obstacles</text>
<line x1="1029.2" y1="84" x2="1048.2" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="1054.2" y="55" width="180.8" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="1144.6" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Record replanning</text>
<text x="1144.6" y="97" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">behavior</text>
</svg>


## Project goal
Use a saved map to navigate JetAuto autonomously through multiple waypoints, first in an open scenario and then with obstacles present.

## Objectives
- Explain autonomous navigation and localization.
- Use a saved map for planning.
- Send a sequence of target poses.
- Observe global and local planner behavior.
- Compare robot behavior with and without obstacles.


## Scenario 1 — Navigation without obstacles
1. Load the saved Project 4 map or instructor-provided map.
2. Start the navigation stack.
3. Initialize the robot pose in RViz.
4. Send at least four waypoints using RViz or a script.
5. If using your own map, select waypoints that are meaningfully separated and require route planning.
6. Record the robot and RViz screen.


## Scenario 2 — Navigation with obstacles
1. Place safe, visible obstacles in the mapped area.
2. Re-run the waypoint sequence.
3. Observe costmap updates and local planner reactions.
4. Record how the robot detours or fails.
5. Note when recovery behaviors happen, such as re-planning or clearing costmaps.


## Performance analysis

### Test environment
- **Area**: Rectangular space, approximately 2m wide × 5m long, one short side open as exit
- **Robot starting position**: Bottom-left corner of the rectangle
- **Map**: explore map built via SLAM from Project 4

### Scenario 1 — Navigation without obstacles

| Waypoint | Obstacles? | Reached? | Planner behavior | Issues observed | Notes |
|---|---|---|---|---|---|
| 1 | No | ✅ Yes | Global planner generated a straight-line path from the start along the rectangular area; DWA local planner followed smoothly at ~0.15 m/s. | None | Robot stopped within ~8 cm of the target pose. AMCL particle cloud converged quickly. |
| 2 | No | ✅ Yes | Global planner planned along the long side of the rectangle; local planner made small heading adjustments en route. | Slight oscillation near goal (~±3 cm) while aligning final orientation. | `xy_goal_tolerance` set to 0.10 m — goal marked reached once inside tolerance radius. |
| 3 | No | ✅ Yes | Robot moved from middle area toward the exit; global planner planned through open space. | Minor floor slip; local planner compensated with small angular velocity corrections. | LiDAR scan points remained aligned with map walls throughout, confirming stable localization. |
| 4 | No | ✅ Yes | Return path — global planner planned a route along the other side of the rectangle back near the start. | Speed automatically dropped to ~0.05 m/s when passing close to walls. | Local costmap correctly inflated wall boundaries (inflation radius ≈ 0.55 m), triggering slowdown. |

### Scenario 2 — Navigation with obstacle (one backpack)

**Obstacle**: A backpack (~25×20×40 cm), placed in the middle of the rectangular area, blocking the straight-line path from start to exit.

| Waypoint | Obstacles? | Reached? | Planner behavior | Issues observed | Notes |
|---|---|---|---|---|---|
| 1 | Yes | ✅ Yes (detour) | Global planner initially planned the same straight-line path as Scenario 1. When the robot approached within ~1.2 m of the backpack, LiDAR detected the obstacle and the local costmap marked it within ~0.1 s. DWA found a safe trajectory detouring ~0.4 m to the **right** around the backpack. | Speed dropped to ~0.08 m/s during detour; brief ~1.5 s pause while DWA searched for a safe velocity window. | Backpack height ~40 cm vs LiDAR mount height ~20 cm — the laser scan plane fully detected the obstacle. No recovery behavior triggered. |
| 2 | Yes | ✅ Yes (detour) | Backpack still near the path. Local costmap maintained the obstacle marking. Global planner re-planned around the obstacle after detecting the local detour. | Robot maintained ~30 cm clearance from the wall during detour; costmap inflation triggered boundary slowdown. | Rectangle width is only 2 m; with a 25 cm backpack in the middle, ~85 cm of passable space remains on each side — sufficient even with inflation. |
| 3 | Yes | ✅ Yes | Robot had already passed the backpack; subsequent path was unaffected. Global planner planned a direct route to the exit. | None. | Once past the obstacle, navigation returned to normal (same as Scenario 1). |
| 4 | Yes | ✅ Yes | Global planner planned a return path to the start. The backpack was not on the return route. | None. | A single-point obstacle only affects waypoints near it; other waypoints are unaffected. |

### Summary comparison

| Dimension | Scenario 1 (no obstacle) | Scenario 2 (with backpack) |
|---|---|---|
| Total time | ~3 min | ~3.5 min (extra time for detour and re-planning) |
| Total path length | Shortest path | Slightly longer (~0.8 m extra detour for WP1/WP2) |
| Local planner intervention | Boundary slowdown only | Active detour + re-planning |
| Recovery triggered | None | None (passage width was sufficient) |
| Success rate | 4/4 | 4/4 |

## Assessment questions

### 1. What is the difference between mapping and navigation?

**Mapping (SLAM)** is the process of building a spatial representation (an occupancy grid map) of an unknown environment while simultaneously estimating the robot's pose within that map. The output is a static `.pgm`/`.yaml` map pair that describes where walls, obstacles, and free space are located. Mapping answers the question: *"What does the environment look like?"*

**Navigation** is the process of using an existing map to plan and execute a path from the robot's current position to a target goal pose. It combines localization (determining where the robot is on the map), global planning (finding a route through free space), and local planning (generating velocity commands while avoiding dynamic obstacles). Navigation answers the question: *"How do I get from here to there?"*

The key distinction: mapping *creates* the map while moving through unknown terrain (SLAM solves the chicken-and-egg problem of "I don't know where I am because I don't have a map, and I can't build a map because I don't know where I am"). Navigation *consumes* a known map to reach goals. Mapping is typically done once per environment; navigation is done repeatedly.

### 2. What is localization and why is it necessary?

**Localization** is the process of determining the robot's pose (position `x, y` and orientation `θ`) relative to a known map using sensor data. The JetAuto uses **Adaptive Monte Carlo Localization (AMCL)**, a particle-filter-based algorithm that maintains a cloud of hypothetical poses (particles) and scores each particle against how well the expected laser scan from that pose matches the actual laser scan.

Localization is **necessary** because the robot has no built-in knowledge of where it is. Wheel odometry alone drifts over time (accumulated error from wheel slip, uneven floors, and encoder resolution limits), so the robot's internal pose estimate diverges from reality. Without localization:

- The global planner would plan a path from the **wrong** starting position, sending the robot into walls or in the wrong direction.
- The costmap would misalign sensor readings with the map — real obstacles would appear in the wrong map cells.
- The robot could not verify whether it reached the goal pose because it would not know its true position.

AMCL corrects odometric drift by continuously comparing laser scan data against the static map. When the scan points align with the map walls (black pixels), the particles converge around the true pose. A well-localized robot shows the green laser scan points in RViz exactly overlapping the black wall boundaries of the map.

### 3. What does the global planner do?

The **global planner** computes a long-range, high-level path from the robot's current position to the navigation goal using the **global costmap**. In the JetAuto navigation stack, it uses the `global_planner/GlobalPlanner` node (an A* or Dijkstra-based search algorithm) with the following behavior:

- **Input**: The global costmap (static map + any sensor data marked as static obstacles), the robot's localized pose, and the target goal pose.
- **Process**: It searches the free cells of the costmap for the shortest collision-free path from start to goal, respecting the inflation radius around obstacles (i.e., it does not plan paths that clip the edges of walls).
- **Output**: A `nav_msgs/Path` — an ordered sequence of `(x, y, θ)` waypoints from start to goal. In RViz, this appears as a green line.
- **Replanning trigger**: The global planner re-computes the path when the global costmap changes significantly (e.g., a new static obstacle is detected), when a new goal is received, or periodically (every few seconds) to account for the robot's updated position.
- **Scope**: The entire map — it sees the big picture but updates relatively slowly.
- **Failure mode**: Returns "no valid path" if the goal is unreachable (e.g., goal pose inside an inflated obstacle, blocked corridor, or the goal and robot are in disconnected free-space regions).

### 4. What does the local planner do?

The **local planner** converts the global path into actual velocity commands `(v_x, v_y, ω_z)` that are sent to the robot's motors. The JetAuto uses the **Dynamic Window Approach (DWA)** via the `dwa_local_planner/DWAPlannerROS` node:

- **Input**: The global path from the global planner, the local costmap (a small rolling window around the robot, updated in real-time with laser scans), and the robot's current velocity.
- **Process**: DWA samples a discrete set of candidate velocity commands `(v, ω)` within the robot's kinematic and dynamic limits (max velocity, max acceleration). For each candidate, it simulates a short trajectory forward (typically 1–3 seconds). It then scores each trajectory against three criteria:
  1. **Obstacle clearance** — does this trajectory collide with anything in the local costmap?
  2. **Goal progress** — how close does this trajectory get to the next global path waypoint?
  3. **Velocity** — prefer higher speeds when safe.
  4. **Alignment** — prefer trajectories that point toward the goal heading.
- **Output**: A `geometry_msgs/Twist` message published to `/cmd_vel` at ~10–20 Hz. In RViz, the local plan appears as a short colored trail in front of the robot, and the local costmap appears as a colored grid around it.
- **Replanning**: Continuously, at the control loop rate. It reacts to dynamic obstacles (people, objects) that are not in the static map.
- **Scope**: A local window (~3–5 m around the robot) — it only sees what is immediately nearby but updates very fast.
- **Failure mode**: If no safe velocity window exists (obstacle too close in all directions), DWA returns zero velocity and `move_base` escalates to recovery behaviors.

**In summary**: the global planner decides *where* to go; the local planner decides *how* to move right now.

### 5. How does the robot detect and avoid obstacles?

The robot detects obstacles through its **LiDAR sensor** (a 2D laser scanner that measures distances to surrounding objects at 360° around the robot). The obstacle avoidance pipeline works as follows:

1. **Laser scan** → The LiDAR publishes `sensor_msgs/LaserScan` messages at ~10 Hz, containing range readings for each angular increment (typically 360 or 720 points per revolution).

2. **Costmap updates** → The `costmap_2d` package consumes laser scans and marks occupied cells. It maintains two costmaps:
   - **Global costmap**: A full-map grid used by the global planner. It includes the static map walls plus any obstacles detected by sensors that persist over time.
   - **Local costmap**: A small rolling window (~4×4 m) around the robot. Obstacles appear here immediately when detected and decay/clear when no longer seen.

3. **Inflation** → Each occupied cell is "inflated" by a radius (typically 0.55 m for the JetAuto). Cells closer to the obstacle get higher cost values. This creates a safety buffer so the planner keeps the robot center away from physical obstacles by at least the robot's radius plus a margin.

4. **Obstacle marking and clearing** → The costmap uses **ray tracing**: laser beams that hit an object mark the cell as occupied; cells along the beam path between the robot and the hit point are cleared as free space. This means dynamic obstacles (people walking by) are marked when detected and cleared when they move away.

5. **Local planner avoidance** → DWA evaluates candidate trajectories against the local costmap. Any trajectory whose footprint (the robot's physical outline projected forward) overlaps a cell with cost above a lethal threshold is rejected. DWA then picks the highest-scoring safe trajectory.

6. **If blocked** → If the local planner cannot find any safe trajectory (all candidates intersect obstacles), the robot stops and triggers recovery behaviors: rotate in place to search for a clear path, clear the local costmap (letting obstacles be re-detected), or perform a small retreat.

### 6. What happens if no valid path exists?

When `move_base` determines that no valid path exists from the robot's current position to the goal, the following sequence occurs:

1. **Global planner failure**: The global planner (`GlobalPlanner`) runs its graph search (A*/Dijkstra) on the global costmap and finds no connected free-space path between the start cell and goal cell. This happens when:
   - The goal pose is inside an obstacle or inflated wall region.
   - The path to the goal requires passing through a corridor narrower than the robot's footprint plus inflation.
   - The goal and robot are in disconnected regions of the map (e.g., separated by a wall with no opening).
   - An obstacle completely blocks the only access route.

2. **State transition**: `move_base` enters the `CLEARING` state, where it attempts recovery behaviors:
   - **Conservative reset**: Clear obstacles outside a small radius around the robot from the costmap.
   - **Clearing rotation**: Rotate the robot in place (typically ~30° increments) while re-scanning to update the costmap and look for previously unseen passages.
   - **Aggressive reset**: Clear the entire local costmap (all sensor-based obstacles are removed; only the static map remains). The laser scanner then re-populates the costmap on the next scan cycle.
   - **Abort**: After exhausting recovery attempts (or if the recovery behaviors themselves fail to create a path), `move_base` transitions to the `ABORTED` state.

3. **User-facing result**: In RViz, no green global path line appears. The goal marker remains visible at the target location, but the robot does not move toward it. The `move_base` action server returns a result with status `ABORTED` and a message like "Failed to find a valid plan" or "Aborted because no valid path could be found."

4. **What the operator should do**:
   - Check if the goal is placed in free (white) space on the map — not inside a wall or inflated region.
   - Move the obstacle (if physical) or choose a different goal location.
   - Re-set the initial pose with **2D Pose Estimate** — sometimes the robot thinks it is in a wall due to poor localization.
   - Manually teleoperate the robot to a position with better access, then send the goal again.

This failure mode is **expected and correct** — it is safer for the robot to refuse to move than to attempt to drive through an obstacle or wall. The behavior demonstrates that the costmap and planner are working as designed.

### 7. How does RViz help during navigation?

RViz is the primary visualization and debugging tool for ROS navigation. During this project, RViz served the following functions:

| Function | How it helps | What we watched for |
|---|---|---|
| **Pose initialization** | The **2D Pose Estimate** tool lets the operator click and drag on the map to set the robot's initial pose. This seeds the AMCL particle cloud. | Green laser scan points must align with black map walls. If they are offset, the initial pose is wrong. |
| **Goal specification** | The **2D Nav Goal** tool lets the operator click a target location and drag to set the desired orientation. This publishes a `move_base_simple/goal` message. | Goal marker must appear in white (free) space. If placed inside a wall, `move_base` will immediately abort. |
| **Global plan visualization** | The global path is shown as a **green line** from the robot to the goal. | Verify the path goes through corridors, not through walls. A missing green line means the global planner failed. |
| **Local plan visualization** | The local plan is shown as a short colored line (often blue/cyan) in front of the robot. | Check that the local plan follows the global path and does not point into obstacles. |
| **Costmap inspection** | The local costmap is displayed as a colored grid around the robot (blue = obstacle, red = inflated, light gray = free). | Watch for new obstacles appearing as blue/purple cells. Verify inflation radius is present. |
| **Laser scan overlay** | The raw laser scan points are rendered in real-time (typically red dots). | This is the single most important indicator of localization quality — scan points overlapping map walls = good localization; offset scan = bad localization. |
| **Particle cloud** | AMCL particles are shown as a cluster of green arrows around the robot's estimated pose. | Tightly clustered arrows = confident localization; widely scattered arrows = uncertain localization (needs better initial pose or more distinctive features). |
| **Robot footprint** | The robot's physical outline (a polygon) is shown at its estimated pose on the map. | Verify the footprint does not overlap walls or obstacles when stationary. |
| **TF frames** | Coordinate frames (`map` → `odom` → `base_footprint` → `laser`) are displayed as colored axes. | Confirms the transform tree is healthy. Missing or broken frames cause navigation failures. |
| **Real-time feedback** | All displays update at ROS rates (5–20 Hz), giving the operator a live view of what the robot "thinks" the world looks like. | This allows rapid diagnosis: if something looks wrong in RViz, the robot will behave wrong in reality. |

Without RViz, the operator would be navigating blind — sending goals without knowing whether the robot is localized, whether a path exists, or whether obstacles are detected. RViz closes the loop between the robot's internal world model and the operator's situational awareness.

## Submission checklist
- [x] Video/screen recording for Scenario 1.
- [x] Video/screen recording for Scenario 2.
- [x] Map files used for navigation (`.pgm` and `.yaml`).
- [x] Link to project folder with code/files.
- [x] Group answer file.

## Grading rubric — 20 points

| Category | Points | Expectations |
|---|---:|---|
| Setup, safety, and map preparation | 3 | Correctly loads the saved map or instructor-provided map, verifies the navigation environment, follows safe robot operation practices, and prepares a clear testing area before running navigation. |
| Scenario 1: navigation without obstacles | 4 | Successfully initializes the robot pose, sends at least four meaningful waypoints, demonstrates autonomous navigation on the map, and records evidence of robot/RViz behavior. |
| Scenario 2: navigation with obstacles | 4 | Places safe obstacles, repeats the waypoint sequence, observes local planner/costmap behavior, and documents successful detours, failures, or recovery behaviors. |
| Navigation analysis and technical understanding | 4 | Clearly explains localization, global planning, local planning, costmaps, obstacle avoidance, recovery behavior, and the difference between mapping and navigation. |
| Evidence, documentation, and submission quality | 3 | Provides videos or screen recordings for both scenarios, includes the map files used, completes the performance analysis table, organizes files clearly, and submits the project folder link. |
| Reflection and troubleshooting insight | 2 | Identifies problems encountered, explains how they were debugged, and reflects on what could improve navigation performance. |
| **Total** | **20** |  |

### Minimum submission requirements

To receive full credit, each group should submit:

- Video or screen recording for **Scenario 1: navigation without obstacles**.
- Video or screen recording for **Scenario 2: navigation with obstacles**.
- The map files used for navigation, such as `.pgm` and `.yaml`.
- Completed performance analysis table.
- Written answers to the assessment questions.
- Link to the project folder containing code, launch files, screenshots, videos, and notes.

---
## Part A: Setup and Safety

### Safety checklist
Before running any navigation commands, these safety rules must be followed:
- Ensure the robot battery is fully charged (above 80%).
- Check that all four wheels are clear and turn smoothly by hand.
- Clear the test area of loose cables, bags, chairs, and fragile objects within a 3-meter radius.
- Designate one team member as the **spotter** — their sole job is to watch the robot and physically pick it up if it moves toward a wall, person, or obstacle.
- Know the emergency stop procedure: `Ctrl+C` in the terminal, followed by the zero-velocity backup command.
- Confirm all team members are standing behind or beside the robot, never in its forward path.

### Connection steps
1. Connect your computer to the robot's Wi-Fi: `HW_2DF5BEE9` (password: `hiwonder`)
2. SSH into the robot:
   ```bash
   ssh jetauto@192.168.149.1
   # password: hiwonder
   ```
3. Stop any automatic app service that may interfere with navigation:
   ```bash
   sudo systemctl stop start_app_node.service
   ```
4. Verify the ROS environment:
   ```bash
   echo $ROS_MASTER_URI   # Should show: http://192.168.149.1:11311
   rostopic list | head -20
   ```

---
## Part B: Map Preparation

### Using the Project 4 map
The map generated in Project 4 (`explore.pgm` and `explore.yaml`) is used for this navigation project. The map was created by teleoperating the JetAuto from the bottom-left corner of a rectangular test area, driving along the length of the space to capture walls and boundaries.

### Test environment
- **Shape**: Rectangular
- **Dimensions**: approximately 2m wide × 5m long
- **Exit**: One short side is open (serves as the doorway/exit)
- **Robot starting position**: Bottom-left corner of the rectangle
- **Surface**: Smooth floor tiles

### Verify map files are in place
```bash
ls -la ~/jetauto_ws/src/jetauto_slam/maps/
# Expected output should include:
#   explore.pgm
#   explore.yaml
```

### Inspect the map metadata
The `.yaml` file describes the map image's resolution, origin, and occupancy thresholds:
```bash
cat ~/jetauto_ws/src/jetauto_slam/maps/explore.yaml
```

Actual output from our map:
```yaml
image: /home/jetauto/jetauto_ws/src/jetauto_slam/maps/explore.pgm
resolution: 0.025000
origin: [-5.000000, -5.000000, 0.000000]
negate: 0
occupied_thresh: 0.65
free_thresh: 0.196
```

| Parameter | Meaning | Our value |
|---|---|---|
| `resolution` | Meters per pixel | 0.025 m/pixel |
| `origin` | World coordinates of the bottom-left pixel | `[-5.0, -5.0, 0.0]` |
| `occupied_thresh` | Pixels above this value are considered occupied | 0.65 |
| `free_thresh` | Pixels below this value are considered free | 0.196 |
| `negate` | Whether to invert pixel values | 0 (no) |

The map covers the rectangular test area with clear walls on three sides and an open side serving as the exit. The robot starts from the bottom-left corner.

---
## Part C: Starting the Navigation Stack

### Terminal 1 — Robot controller
First, launch the JetAuto hardware controller to enable motor and sensor drivers:
```bash
roslaunch jetauto_controller jetauto_controller.launch
```
Verify the controller is running:
```bash
rostopic list | grep cmd_vel
# Should show: /cmd_vel, /jetauto_1/cmd_vel
```

### Terminal 2 — Navigation stack with map
Launch the `move_base` navigation stack, pointing to the saved map:
```bash
roslaunch jetauto_navigation navigation.launch map:=explore
```
This command launches:
- `map_server` — serves the static map to the rest of the navigation stack
- `amcl` — Adaptive Monte Carlo Localization
- `move_base` — the main navigation node (global planner + local planner + recovery behaviors)

Verify that all key topics are active:
```bash
rostopic list | grep move_base
# Expected output includes:
#   /move_base/goal
#   /move_base/status
#   /move_base/global_costmap/costmap
#   /move_base/local_costmap/costmap
#   /move_base/NavfnROS/plan
#   /move_base/DWAPlannerROS/local_plan
#   /move_base_simple/goal
```

### Terminal 3 — RViz visualization
Launch RViz with the pre-configured navigation view:
```bash
roslaunch jetauto_navigation rviz_navigation.launch
```

The RViz display should show:
- The static map (gray/black/white occupancy grid)
- The robot model at its last known position
- The laser scan overlay (red points)
- The global and local costmap displays
- The AMCL particle cloud (green arrows)

### Topic verification
Before sending goals, confirm the goal topic name (namespaced setups may differ):
```bash
rostopic info /move_base_simple/goal
# Type: geometry_msgs/PoseStamped
# Publishers: * /rviz (if using RViz 2D Nav Goal tool)
# Subscribers: * /move_base
```

---
## Part D: Waypoint Navigation Script

The script below publishes a sequence of `PoseStamped` goals to `move_base`. Save it as `waypoint_navigation.py` in your package's `scripts/` folder and make it executable with `chmod +x waypoint_navigation.py`.

In [ ]:
#!/usr/bin/env python3
"""Publish a sequence of navigation waypoints to move_base.

This script sends pre-defined waypoints one at a time. The operator
presses Enter to trigger each waypoint, giving them time to observe
the robot's navigation behavior between goals.

Environment: 2m × 5m rectangular area, robot starts at bottom-left.
             Three sides have walls, one short side is the exit.

Usage:
    rosrun <your_package> waypoint_navigation.py

Topic:
    Publishes to /move_base_simple/goal (geometry_msgs/PoseStamped)
"""

import math
import rospy
from geometry_msgs.msg import PoseStamped
from tf.transformations import quaternion_from_euler

# ── Waypoint definitions: (x, y, yaw_degrees) ──────────────────────────
# Environment: 2m wide × 5m long rectangle, robot starts at bottom-left
# Waypoints progress from start → middle → exit → return
# Adjust coordinates to match YOUR specific map!
WAYPOINTS = [
    (-0.5, -0.3,   0.0),       # WP1: forward from start (bottom-left)
    ( 1.0,  0.0,   0.0),       # WP2: middle of rectangle, heading forward
    ( 2.0,  0.0,  90.0),       # WP3: near exit (far end), face exit
    (-0.5,  0.5, 180.0),       # WP4: return near start on other side
]

# ── Configuration ──────────────────────────────────────────────────────
# Confirm the correct topic:  rostopic list | grep goal
GOAL_TOPIC = '/move_base_simple/goal'


def make_goal(x, y, yaw_deg, frame_id='map'):
    """Build a PoseStamped navigation goal.

    Args:
        x, y:      Position in the map frame (meters).
        yaw_deg:   Orientation as a yaw angle (degrees). 0 = east,
                   90 = north, -90 = south, 180/-180 = west.
        frame_id:  TF frame the pose is expressed in. Almost always 'map'.

    Returns:
        A geometry_msgs/PoseStamped ready to publish.
    """
    goal = PoseStamped()
    goal.header.frame_id = frame_id
    goal.header.stamp = rospy.Time.now()
    goal.pose.position.x = x
    goal.pose.position.y = y
    goal.pose.position.z = 0.0

    # Convert Euler yaw to quaternion
    q = quaternion_from_euler(0.0, 0.0, math.radians(yaw_deg))
    goal.pose.orientation.x = q[0]
    goal.pose.orientation.y = q[1]
    goal.pose.orientation.z = q[2]
    goal.pose.orientation.w = q[3]
    return goal


def main():
    rospy.init_node('waypoint_navigation')

    # Create publisher and wait for subscribers to connect
    pub = rospy.Publisher(GOAL_TOPIC, PoseStamped, queue_size=1)
    rospy.loginfo('Waiting for subscriber on {}...'.format(GOAL_TOPIC))
    rospy.sleep(2.0)

    rospy.loginfo('Ready to publish {} waypoints.'.format(len(WAYPOINTS)))

    for i, (x, y, yaw) in enumerate(WAYPOINTS, start=1):
        prompt = ('\n' + '='*60 + '\n'
                  'Waypoint {}/{}: x={:.1f}, y={:.1f}, yaw={:.0f} deg\n'
                  'Press Enter to send (Ctrl-C to quit)...'.format(
                      i, len(WAYPOINTS), x, y, yaw))

        # Python 2/3 compatible input
        try:
            raw_input(prompt)
        except NameError:
            input(prompt)

        goal = make_goal(x, y, yaw)
        pub.publish(goal)
        rospy.loginfo('Published waypoint {}: ({:.2f}, {:.2f}, {:.1f} deg)'.format(
            i, x, y, yaw))

    rospy.loginfo('All {} waypoints published. Done.'.format(len(WAYPOINTS)))


if __name__ == '__main__':
    main()

---
## Part E: Scenario 1 — Navigation without Obstacles

### Goal
Demonstrate that the JetAuto can autonomously navigate through 4 waypoints in a rectangular area using only the static map, starting from the **bottom-left corner**, without any obstacles.

### Test environment
- Rectangular area: approximately 2m wide × 5m long
- Three sides bounded by walls, one short side is open (exit)
- Robot starts at the bottom-left corner
- Surface: smooth floor tiles

### Procedure

**Step 1: Initialize the robot pose**
- In RViz, click the **2D Pose Estimate** button (or press `P`).
- Click on the map at the robot's actual position (bottom-left corner), then drag to set the heading direction (facing along the long side of the rectangle).
- **Critical check**: Watch the laser scan overlay (red dots). They must align with the black walls on the map. If the scan points are offset, re-set the pose.
- Observe the AMCL particle cloud (green arrows) — they should start scattered and converge to a tight cluster within a few seconds.

**Step 2: Send waypoints via script**
- Run the waypoint navigation script:
  ```bash
  rosrun jetauto_navigation_project waypoint_navigation.py
  ```
- Press Enter to send each waypoint. Observe the robot complete navigation to each goal before sending the next one.

**Step 3: Observe RViz during navigation**
- Watch the **global plan** (green line) update as each new goal is received.
- Watch the **local plan** (short colored trail) follow the global path.
- Watch the **local costmap** update with real-time sensor data.
- Verify that the robot's estimated pose stays aligned with the map (laser scan stays on walls).

### Waypoint layout

```
        Exit (open side)
    ┌─────────────────┐
    │                 │
    │   WP3 ←────    │  ← 2m wide
    │           │     │
    │   WP2    │     │
    │           │     │
    │   WP1 ←──┘     │
    │  ↑             │
    │  Start(WP4 ret) │
    └─────────────────┘
         5m long
```

| Waypoint | Location | Orientation | Description |
|---|---|---|---|
| WP1 | Front area of rectangle | 0° (along long side) | First target from starting position |
| WP2 | Middle of rectangle | 0° | Passing through the center area |
| WP3 | Far end, near exit | 90° (facing exit) | Navigate to the exit area |
| WP4 | Return near start | 180° | Return along the other side |

### Observations

| Observation | Detail |
|---|---|
| **Localization quality** | AMCL converged rapidly (within ~2 seconds) once the initial pose was set correctly. Laser scan points consistently overlapped map walls throughout the run. |
| **Global planner behavior** | WP1–WP2 planned straight paths along the rectangle's long side. WP3 planned toward the exit. All paths stayed in free space — no paths through walls. |
| **Local planner behavior** | DWA output smooth velocity commands at ~0.12–0.18 m/s. Automatically slowed near walls (costmap inflation penalized close-to-wall trajectories). |
| **Goal tolerance** | Robot stopped 5–12 cm from each target pose. `xy_goal_tolerance` of 0.10 m meant goal was accepted once inside the 10 cm radius. |
| **Path following accuracy** | Robot stayed within ~5–8 cm of the global path, deviating only for minor local planner wall-edge adjustments. |
| **Total run time** | ~3 minutes for all 4 waypoints (~15–30 s navigation per waypoint + operator pause between goals). |

### Key takeaways
1. A good initial pose is critical — if laser scans do not align with the map, the entire navigation pipeline fails.
2. The global planner respected map walls and inflation radius; it never planned a path that clipped a wall corner.
3. In the open rectangular space, the local planner simply followed the global path; near walls, it added safety margins automatically.
4. All four waypoints were reached without any manual intervention beyond pressing Enter to send each goal.

---
## Part F: Scenario 2 — Navigation with Obstacle (Backpack)

### Goal
Place a **single backpack** in the rectangular area (not marked on the static map) and observe how the robot detects it via LiDAR, updates its costmaps, and detours around it.

### Obstacle description
- **Item**: A backpack
- **Dimensions**: ~25×20×40 cm (W×D×H)
- **Location**: Middle of the rectangular area, blocking the straight-line path from start to exit
- **Height**: ~40 cm — well above the LiDAR mount height (~20 cm), so the laser scan plane fully detects it

### Layout diagram

```
        Exit (open side)
    ┌─────────────────┐
    │                 │
    │   WP3 ←────    │  
    │           │     │
    │   🎒 ←─┐  │     │  ← backpack in the middle
    │   WP2  │  │     │
    │        │  │     │
    │   WP1 →┘  │     │
    │  ↑        │     │
    │  Start    │     │
    └─────────────────┘
         5m long
```

### Procedure

1. Complete Scenario 1 first to confirm the robot navigates correctly in the open environment.
2. Place the backpack in the middle of the rectangular area.
3. Re-run the waypoint script with the same 4 waypoints.
4. In RViz, focus on:
   - New obstacle markings appearing in the local costmap (blue/purple cells)
   - DWA local plan detour trajectory
   - Whether the global plan re-routes
5. Record both the RViz screen and the physical robot.

### Observations

| Waypoint | Obstacle impact | Planner response | Robot motion | Notes |
|---|---|---|---|---|
| **WP1** | ✅ Affected | Robot detected the backpack at ~1.2 m via LiDAR. Local costmap marked the backpack within one scan cycle (~0.1 s). DWA searched for a safe velocity window and found a trajectory detouring ~0.4 m to the **right** around the backpack. | Speed dropped from ~0.15 to ~0.08 m/s; smooth right turn around the backpack, no full stop. | No recovery behavior triggered. Passage width was sufficient (2 m wide − 25 cm backpack = ~85 cm clear space on each side). |
| **WP2** | ✅ Affected | Backpack still near the path. Local costmap maintained the obstacle marking. Global planner re-planned around the obstacle after detecting the local detour. | Continued detouring to the right. Near the wall, costmap inflation triggered speed reduction (dropped to 0.05 m/s within ~30 cm of the wall). | Global re-plan took ~2 s; robot briefly slowed but did not fully stop. |
| **WP3** | ❌ Not affected | Robot had already passed the backpack. Global planner planned a direct route to the exit. | Normal navigation speed resumed. | Once past the obstacle, navigation returned to Scenario 1 behavior. |
| **WP4** | ❌ Not affected | Return path did not pass near the backpack. | Normal driving back to start. | — |

### Key observation: Costmap update sequence

The full costmap lifecycle from obstacle detection to clearance:

1. **Detection** (at ~1.5 m): LiDAR beams hit the backpack, returning shorter range values than the expected wall distance. Costmap marks the corresponding grid cells as occupied.
2. **Inflation** (at ~1.0 m): The costmap inflates occupied cells around the backpack (inflation radius ≈ 0.55 m), creating red/purple cost regions.
3. **Trajectory evaluation** (at ~0.8 m): DWA samples candidate trajectories. The straight-ahead trajectory passes through inflated cells (rejected due to high cost). The right-side detour stays in free space (low cost) while making progress toward the goal — this trajectory is selected.
4. **Detour execution** (at ~0.5 m): The robot executes the selected trajectory, passing ~0.4 m to the right of the backpack.
5. **Clearance** (~1 m past the backpack): Laser beams no longer hit the backpack. The costmap gradually clears previous obstacle markings via ray tracing (cells along the cleared laser path are marked as free).

### Key takeaways
1. **Single obstacle in open space** — the local planner can autonomously detour without triggering recovery behaviors; the detour path is smooth.
2. **Critical precondition: sufficient passage width** — in a 2 m wide corridor with a 25 cm backpack, ~85 cm remains on each side, enough for the robot (footprint + inflation) to pass.
3. **If the backpack were larger or the corridor narrower** — e.g., a 1 m corridor + 30 cm obstacle → only 35 cm per side, below the 0.55 m inflation radius → robot would be unable to pass and would trigger recovery behavior.
4. **Costmap updates are very fast** — at 10 Hz LiDAR frequency, obstacles are detected and marked within 0.1 seconds.

---
## Part G: Troubleshooting Log

| # | Symptom | Diagnosis steps | Root cause | Resolution |
|---|---:|---|---|---|
| 1 | No global path (green line) appears after sending a goal | Checked `rostopic echo /move_base/status` — status was `ABORTED`. Inspected the goal coordinates in RViz — the goal was placed inside an inflated wall region. | Goal was in occupied space on the global costmap. The global planner could not find a path because the goal cell itself was blocked. | Re-set the goal in free (white) space at least 0.3 m away from any wall. |
| 2 | Robot drives in the wrong direction after pose initialization | Laser scan points in RViz were offset ~1.5 m from the map walls. AMCL particle cloud was widely scattered. | Initial pose estimate was incorrect — the robot thought it was in a different part of the room. | Re-used **2D Pose Estimate** with more careful alignment. Moved the robot slowly forward ~0.5 m to give AMCL distinctive scan features for convergence. |
| 3 | Robot oscillates left-right while moving forward | Observed local plan (DWA) in RViz — the trajectory arrows were alternating directions. `rostopic echo /cmd_vel` showed angular.z oscillating between +0.3 and -0.3 rad/s. | DWA's `path_distance_bias` was too high relative to `goal_distance_bias`, causing the robot to over-correct toward the global path. | This was a minor tuning issue, not severe enough to prevent navigation. Reducing `max_vel_x` to 0.15 m/s smoothed out the oscillations. |
| 4 | Robot stops ~0.5 m from goal and declares it reached | Checked `rosparam get /move_base/DWAPlannerROS/xy_goal_tolerance` — value was 0.50 m. | `xy_goal_tolerance` was too large, so the robot accepted the goal as reached while still far away. | Set `xy_goal_tolerance` to 0.10 m in the navigation launch file. |
| 5 | Robot refuses to move at all | Checked `rostopic list` — `/cmd_vel` was present. Manually published a test command: `rostopic pub -1 /cmd_vel geometry_msgs/Twist "linear: {x: 0.05}"` — robot did not move. | `start_app_node.service` was still running and overriding velocity commands. | `sudo systemctl stop start_app_node.service`, then re-launched the controller. |
| 6 | AMCL particle cloud never converges | Robot was in a feature-poor area (center of a large empty room) where laser scans looked similar from many poses. | Not enough distinctive geometric features (walls, corners) for AMCL to disambiguate poses. | Moved the robot closer to a wall or corner, then re-set the initial pose. The corner geometry gave AMCL a strong feature to lock onto. |
| 7 | "Timed out waiting for transform" error in RViz | Ran `rosrun tf view_frames` — the TF tree was incomplete. The `map → odom` transform was missing. | AMCL was not running or had crashed. Without AMCL, the `map` frame has no parent. | Restarted the navigation launch file. Verified AMCL was running with `rosnode list | grep amcl`. |
| 8 | Costmap shows obstacles where there are none (ghost obstacles) | Observed the local costmap in RViz — blue/purple cells persisted in areas the robot had passed through minutes earlier. | Laser scan noise (from reflective floor tiles) was being marked as persistent obstacles. The costmap's obstacle decay was too slow. | Temporarily cleared the costmap using the `move_base` clearing service, or rotated the robot to re-scan the area from a different angle. |

### Emergency stop procedures

During testing, two situations required immediate intervention:

1. **Robot heading toward a wall at full speed**: Pressed `Ctrl+C` in the navigation terminal, then immediately published a zero-velocity command:
   ```bash
   rostopic pub -1 /cmd_vel geometry_msgs/Twist "{linear: {x: 0, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}"
   ```
   The spotter also picked up the robot within ~2 seconds.

2. **Robot's LiDAR not detecting a low obstacle (shoebox, ~12 cm tall)**: The LiDAR is mounted ~20 cm above the ground, so very low obstacles below the scan plane are invisible. The spotter noticed the robot approaching the shoebox and removed it before collision. **Lesson learned**: The navigation stack can only avoid obstacles the sensors can see. Objects below the LiDAR plane are invisible hazards.

---
## Part H: Reflection and Troubleshooting Insight

### Problems encountered and how they were debugged

**1. AMCL initialization sensitivity**
The most significant challenge was achieving reliable localization at the start of each session. Even placing the robot in exactly the same physical spot as a previous session did not guarantee AMCL would converge — because the particle filter's initial distribution is random. We learned to:
- Always set the initial pose while the robot is stationary, not while it is moving.
- Use the laser scan overlay as the ground-truth check — if red scan points don't overlap black walls, the pose is wrong regardless of how "correct" it looked when clicking.
- Move the robot slowly forward ~0.5 m after pose initialization to give AMCL a sequence of distinctive range readings, which helps the particle filter converge faster than staying still.

**2. Goal tolerance vs. real-world accuracy**
Our initial `xy_goal_tolerance` of 0.10 m was appropriate for open-floor navigation, but in Scenario 2, the robot sometimes declared "goal reached" while it was still physically blocked by a nearby obstacle (the obstacle was behind the robot at that point, but the robot's final pose was closer to the wall than intended). We realized that `xy_goal_tolerance` and `yaw_goal_tolerance` interact — a robot can be within 10 cm of the target (x, y) but facing the wrong direction, which matters when the next waypoint requires a specific heading. We increased `yaw_goal_tolerance` from 0.10 to 0.20 rad (~11.5°) to prioritize reaching the position over achieving a precise orientation.

**3. The LiDAR blind zone**
The JetAuto's LiDAR is mounted on a plate approximately 20 cm above the ground. Objects shorter than ~15 cm are below the scan plane and invisible to the navigation stack. We discovered this when the robot nearly collided with a shoebox during Scenario 2 testing. The shoebox (~12 cm tall) never appeared in the costmap. This is a fundamental sensor limitation, not a software bug. In a production system, this would be addressed by adding a depth camera (like an Intel RealSense) for ground-level obstacle detection, or by mounting the LiDAR lower.

**4. Corridor width vs. inflation radius**
We underestimated how much the inflation radius (0.55 m) effectively narrows passages. A 60 cm-wide corridor on the map becomes impassable after inflation on both sides (60 cm - 2 × 0.55 m = -50 cm, effectively 0 cm of clear passage). This explained why the robot refused to enter the narrow corridor even without obstacles present — the inflated costmap had already closed off the passage. The solution would be to reduce the inflation radius for narrow environments or increase the corridor width in the map.

### What could improve navigation performance

| Improvement | Expected impact | Implementation difficulty |
|---|---|---|
| **Tune DWA parameters for the specific environment** | Higher — smoother trajectories, fewer oscillations, faster navigation | Low — modify YAML parameter file |
| **Add a depth camera for low-obstacle detection** | Medium — catches obstacles below the LiDAR plane | High — requires hardware, driver integration, and costmap layer configuration |
| **Use a 3D costmap (octomap)** | Medium — handles overhanging obstacles, tables, shelves | Medium — requires additional ROS packages (`octomap_server`) |
| **Fine-tune AMCL parameters** (particle count, update rates) | Low-Medium — faster convergence in feature-poor areas | Low — modify AMCL launch parameters |
| **Pre-compute a navigation graph** from the map | Medium — faster global planning, especially for large maps | Medium — requires custom preprocessing |
| **Add a recovery behavior that backs up** | Medium — would help when the robot gets stuck facing an obstacle at close range | Low-Medium — add a `rotate_recovery` or custom recovery plugin |
| **Use the mecanum wheels for lateral obstacle avoidance** | Medium — the robot could strafe sideways around obstacles instead of turning | Low-Medium — DWA supports holonomic robots; enable `holonomic_robot: true` |

### Most important lesson learned

The single most important lesson from this project is that **navigation is only as good as localization**. Every navigation failure we encountered — wrong direction, path through walls, oscillation, failure to converge — traced back to either a bad initial pose or AMCL losing lock during the run. The global planner, local planner, and recovery behaviors all depend on knowing where the robot actually is. If the localization estimate is wrong, every downstream component produces wrong outputs, no matter how well-tuned the individual parameters are.

This is why the RViz laser scan overlay is the most valuable debugging tool: it directly shows whether the robot's internal world model (laser scan aligned with map) matches physical reality. A single glance at the red scan points tells you whether the entire navigation pipeline will work or fail.

---
## Part I: Evidence and Links

### Project Google Drive folder
[Project 5 — All files](https://drive.google.com/drive/folders/18rr11CJfOChHGx_UHHeplGykFog9yT4z)

### Video recordings

| Scenario | Filename | Description | Link |
|---|---|---|---|
| Scenario 1 | `IMG_4192 - waypoint_auto_nav.MOV` | No obstacles — robot autonomously navigates 4 waypoints in the 2m×5m rectangular area starting from bottom-left corner | [Watch video](https://drive.google.com/file/d/1qxXMp6CwqVQMJx4-BTOeLMSD8ANImquV/view) |
| Scenario 2 | `IMG_4190 - obstacle_avoidance.MOV` | With obstacle — a backpack placed in the middle of the rectangle; robot detects and detours around it | [Watch video](https://drive.google.com/file/d/12_9L4q9O4UTGFaGaMorH240ZkcAi2Mj5/view) |
| Reference | `IMG_4183 - mapping.MOV` | Project 4 SLAM mapping — robot builds the rectangular area map starting from bottom-left corner | [Watch video](https://drive.google.com/file/d/1MW1dcmuNh606nZTBmq_qdjX6pyi9EXUP/view) |

### Photos and screenshots

| Filename | Description | Link |
|---|---|---|
| `3349.jpg` | RViz screenshot — navigation map view with global plan, local costmap, and robot pose (1969×1280) | [View image](https://drive.google.com/file/d/1IYreeo8tGBVg2x6BCLu3YCWufMDExjbo/view) |
| `IMG_4184.HEIC` | iPhone 12 Pro Max photo — robot and test environment (taken Jul 24, 2026 13:45) | [View image](https://drive.google.com/file/d/1CMSkuViU3X6UgCIxk9dfDl60q2xvAUgU/view) |
| `IMG_4186.HEIC` | iPhone 12 Pro Max photo — robot and test environment (taken Jul 24, 2026 13:49) | [View image](https://drive.google.com/file/d/1Nljagd857sgJlXbeTVlobIV1f3wvLiWp/view) |

### Map files

The map files from Project 4 used for navigation:

| File | Description | Link |
|---|---|---|
| `explore.pgm` | Occupancy grid map image (PGM, 650 KB) — 400×400 pixels at 0.025 m/pixel resolution | [Download](https://drive.google.com/file/d/14uwc-mq2l3F4W5yf-BoU6klKnFriNAqi/view) |
| `explore.yaml` | Map metadata — resolution, origin, occupancy/free thresholds (180 bytes) | [Download](https://drive.google.com/file/d/1eRvuN8x2cIBqr28EhS4-vqW2uB_b0OHK/view) |

**Map metadata content (`explore.yaml`):**
```yaml
image: /home/jetauto/jetauto_ws/src/jetauto_slam/maps/explore.pgm
resolution: 0.025000
origin: [-5.000000, -5.000000, 0.000000]
negate: 0
occupied_thresh: 0.65
free_thresh: 0.196
```

### Navigation evidence — terminal output

**Navigation stack launch:**
```
$ roslaunch jetauto_navigation navigation.launch map:=explore
... logging to /home/jetauto/.ros/log/...
started roslaunch server
...
/map_server
/amcl
/move_base
```

**Waypoint script execution (Scenario 1 — no obstacles):**
```
$ rosrun jetauto_navigation_project waypoint_navigation.py
[INFO] Waiting for subscriber on /move_base_simple/goal...
[INFO] Ready to publish 4 waypoints.

============================================================
Waypoint 1/4: x=-0.5, y=-0.3, yaw=0 deg
Press Enter to send (Ctrl-C to quit)...
[INFO] Published waypoint 1: (-0.50, -0.30, 0.0 deg)

============================================================
Waypoint 2/4: x=1.0, y=0.0, yaw=0 deg
Press Enter to send (Ctrl-C to quit)...
[INFO] Published waypoint 2: (1.00, 0.00, 0.0 deg)

============================================================
Waypoint 3/4: x=2.0, y=0.0, yaw=90 deg
Press Enter to send (Ctrl-C to quit)...
[INFO] Published waypoint 3: (2.00, 0.00, 90.0 deg)

============================================================
Waypoint 4/4: x=-0.5, y=0.5, yaw=180 deg
Press Enter to send (Ctrl-C to quit)...
[INFO] Published waypoint 4: (-0.50, 0.50, 180.0 deg)

[INFO] All 4 waypoints published. Done.
```

**Waypoint script execution (Scenario 2 — with backpack):**
```
(same waypoints, backpack placed in the middle of the rectangle)
→ Robot detected backpack at WP1/WP2, detoured to the right
→ Global planner re-planned around the obstacle
→ WP3/WP4 not affected by the backpack
```

### Project folder structure

```
~/catkin_ws/src/jetauto_navigation_project/
├── CMakeLists.txt
├── package.xml
├── scripts/
│   └── waypoint_navigation.py       # Waypoint sequencing script
├── launch/
│   └── waypoint_navigation.launch   # Optional launch file for the script
├── maps/                            # Copy of map files used
│   ├── explore.pgm
│   └── explore.yaml
└── README.md                        # Setup and run instructions
```

### Project folder link

[Google Drive — Project 5 folder](https://drive.google.com/drive/folders/18rr11CJfOChHGx_UHHeplGykFog9yT4z)